# 第二课：Token、Embedding 与位置信息

文本如何切分？编号为什么映射成向量？顺序怎样进入计算？按顺序运行以下实验，代码只用 Python 标准库。

## 1. 比较切分单位

六个汉字可以按字、按词或按子词表示。这里的四词切分仅用作对照。

In [1]:
text = "我喜欢喝牛奶"
print("按字：", list(text))
print("一种词切分：", ["我", "喜欢", "喝", "牛奶"])

按字： ['我', '喜', '欢', '喝', '牛', '奶']
一种词切分： ['我', '喜欢', '喝', '牛奶']


## 2. 读懂一次合并

遇到指定的相邻片段，就合成一项；从左到右扫描，不跨句子合并。

In [2]:
def merge_pair(parts, pair):
    result = []
    i = 0
    while i < len(parts):
        if tuple(parts[i:i + 2]) == pair:
            result.append(parts[i] + parts[i + 1])
            i += 2
        else:
            result.append(parts[i])
            i += 1
    return result

## 3. 应用固定规则，再查 ID

规则来自 optional/train_bpe.py 的小语料训练。按顺序观察合并。改输入为我喜欢喝酒，保留规则再运行。这里只覆盖词表中的基础字符，不代表某个商用模型的切分。

In [3]:
merges = [("喜", "欢"), ("喜欢", "喝"), ("喜欢喝", "牛"), ("喜欢喝牛", "奶")]
parts = list(text)
for pair in merges:
    parts = merge_pair(parts, pair)
    print("应用规则", pair, "→", parts)
vocab = ["你", "喝", "喜", "奶", "我", "欢", "牛", "酒"]
vocab += [left + right for left, right in merges]
token_ids = [vocab.index(token) for token in parts]
print("最终 token：", parts)
print("token ID：", token_ids)

应用规则 ('喜', '欢') → ['我', '喜欢', '喝', '牛', '奶']
应用规则 ('喜欢', '喝') → ['我', '喜欢喝', '牛', '奶']
应用规则 ('喜欢喝', '牛') → ['我', '喜欢喝牛', '奶']
应用规则 ('喜欢喝牛', '奶') → ['我', '喜欢喝牛奶']
最终 token： ['我', '喜欢喝牛奶']
token ID： [4, 11]


## 4. 字符、字节、token

牛奶是两个汉字、六个 UTF-8 字节。六个基础字节仍可合并；它们不是六个最终 token 的保证。

In [4]:
word = "牛奶"
print("字符数：", len(word), "UTF-8 字节数：", len(word.encode("utf-8")))
print("UTF-8 字节：", list(word.encode("utf-8")))

字符数： 2 UTF-8 字节数： 6
UTF-8 字节： [231, 137, 155, 229, 165, 182]


## 5. ID 查 embedding 表

沿用同一份12项词表，查 ID 4、11 两行。四维向量采用固定随机种子初始化，尚未学习语义。

In [5]:
import random

vocab = ["你", "喝", "喜", "奶", "我", "欢", "牛", "酒",
         "喜欢", "喜欢喝", "喜欢喝牛", "喜欢喝牛奶"]
rng = random.Random(7)
table = [[round(rng.uniform(-0.5, 0.5), 3) for _ in range(4)] for _ in vocab]
token_ids = [4, 11]
vectors = [table[token_id] for token_id in token_ids]
for token_id, vector in zip(token_ids, vectors):
    print("ID", token_id, vocab[token_id], "→", vector)
print("词表大小 / 向量维度：", len(table), len(table[0]))

ID 4 我 → [0.127, 0.448, 0.077, -0.103]
ID 11 喜欢喝牛奶 → [-0.256, 0.074, 0.025, 0.375]
词表大小 / 向量维度： 12 4


## 6. 重复查表

参数固定时，同一 ID 的输入 embedding 相同；位置和上下文可使后续表示发生变化。

In [6]:
print("再次查‘我’：", table[4])
print("两次查询相同：", vectors[0] == table[4])
print("这是随机初始化，不是训练好的词义表示。")

再次查‘我’： [0.127, 0.448, 0.077, -0.103]
两次查询相同： True
这是随机初始化，不是训练好的词义表示。


## 7. 构造位置编码

D=4 时两组角度分别是 p 和 p/100。位置从0开始，指 token 位置，不是字符偏移。

In [7]:
import math

def position_vector(position, width=4):
    result = []
    for i in range(width // 2):
        angle = position / (10000 ** (2 * i / width))
        result += [math.sin(angle), math.cos(angle)]
    return result

for position in range(3):
    print("位置", position, "→", [round(v, 4) for v in position_vector(position)])

位置 0 → [0.0, 1.0, 0.0, 1.0]
位置 1 → [0.8415, 0.5403, 0.01, 1.0]
位置 2 → [0.9093, -0.4161, 0.02, 0.9998]


## 8. 内容相同，位置不同

这里复制了前面‘我’的四维向量，分别加入位置0和2的编码。验证输入改变，不把这个结果说成模型已经学会正确理解顺序。

In [8]:
word_vector = [0.127, 0.448, 0.077, -0.103]  # embeddings.py 中 ID=4 的一行
combined = []
for position in (0, 2):
    pe = position_vector(position)
    vector = [word_vector[i] + pe[i] for i in range(4)]
    combined.append(vector)
    print("‘我’在位置", position, "→", [round(v, 4) for v in vector])
print("token 向量相同，组合后的输入相同吗？", combined[0] == combined[1])

‘我’在位置 0 → [0.127, 1.448, 0.077, 0.897]
‘我’在位置 2 → [1.0363, 0.0319, 0.097, 0.8968]
token 向量相同，组合后的输入相同吗？ False


## 9. 从切分结果连接到 Attention 输入

使用前面的实际切分结果，重新查 ID、取表行，再为位置0、1等构造输入。修改第一个单元的 text，并从头运行，可观察 token 数和矩阵形状一起改变。这里只演示相加式位置机制。

In [9]:
input_ids = [vocab.index(piece) for piece in parts]
input_vectors = [table[token_id] for token_id in input_ids]
model_inputs = []
for p, vector in enumerate(input_vectors):
    pe = position_vector(p)
    model_inputs.append([vector[i] + pe[i] for i in range(4)])
print("实际切分与 ID：", parts, input_ids)
print("送入后续计算的输入：", [[round(v, 4) for v in row] for row in model_inputs])
print("形状：", [len(model_inputs), len(model_inputs[0])])

实际切分与 ID： ['我', '喜欢喝牛奶'] [4, 11]
送入后续计算的输入： [[0.127, 1.448, 0.077, 0.897], [0.5855, 0.6143, 0.035, 1.375]]
形状： [2, 4]


## 复习

1. 本例为什么会合并喜欢喝牛奶？
2. ID 的数值距离为什么不能解释为语义距离？
3. 同一 token 在两个位置的原始 embedding 与组合输入分别相同吗？

[学生讲义](student_handout.md) · [拓展实验](optional/README.md)